# Pelatihan Model Naïve Bayes

### Deskripsi Tugas
Menggunakan **data training hasil TF-IDF dari Najmi** (file `.npz` dan `.csv`) untuk melatih model **Naïve Bayes**.

Algoritma ini menghitung probabilitas:
> *"Kalau muncul kata X, Y, Z — seberapa besar kemungkinan keputusan pembeliannya True/False?"*

**Input (dari Najmi):**
- `X_train.npz` — matriks TF-IDF data training (7683 × 5000)
- `X_test.npz`  — matriks TF-IDF data testing  (1921 × 5000)
- `y_train.csv` — label training
- `y_test.csv`  — label testing

**Output (dari Muthia):**
- Model Naïve Bayes terbaik tersimpan di `model_naive_bayes.pkl`


## 1. Import Library

In [ ]:
import pickle
import scipy.sparse as sp
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

print("Semua library berhasil diimport!")


Semua library berhasil diimport!


## 2. Load Data dari Najmi

Load file hasil TF-IDF yang sudah disiapkan oleh Najmi.  
Pastikan keempat file berada di folder yang sama dengan notebook ini.


In [ ]:
# Load matriks TF-IDF (sparse matrix)
X_train = sp.load_npz("X_train.npz")
X_test  = sp.load_npz("X_test.npz")

# Load label
y_train_df = pd.read_csv("y_train.csv")
y_test_df  = pd.read_csv("y_test.csv")

print("=== DATA BERHASIL DILOAD ===")
print(f"X_train : {X_train.shape}  (sampel × fitur TF-IDF)")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train_df.shape}")
print(f"y_test  : {y_test_df.shape}")
print()
print("Kolom label:", y_train_df.columns.tolist())
print()
print("Distribusi label y_train (raw):")
print(y_train_df.iloc[:,0].value_counts())


=== DATA BERHASIL DILOAD ===
X_train : (7683, 5000)  (sampel × fitur TF-IDF)
X_test  : (1921, 5000)
y_train : (7683, 1)
y_test  : (1921, 1)

Kolom label: ['purchase_decision']

Distribusi label y_train (raw):
purchase_decision
Y     6794
N      869
y       19
 Y       1
Name: count, dtype: int64


## 3. Bersihkan & Standardisasi Label

Ada variasi penulisan di kolom label (`y`, ` Y` dengan spasi) — perlu diseragamkan dulu.


In [ ]:
def bersihkan_label(series):
    """Standardisasi label: strip spasi, uppercase, lalu encode Y=1 / N=0."""
    bersih = series.astype(str).str.strip().str.upper()
    # Mapping: Y → 1, selain itu → 0
    return (bersih == 'Y').astype(int)

y_train = bersihkan_label(y_train_df.iloc[:, 0])
y_test  = bersihkan_label(y_test_df.iloc[:, 0])

print("Setelah standardisasi:")
print(f"y_train — Beli (1): {y_train.sum():,}  |  Tidak Beli (0): {(y_train==0).sum():,}")
print(f"y_test  — Beli (1): {y_test.sum():,}   |  Tidak Beli (0): {(y_test==0).sum():,}")
print()
print(f"Proporsi kelas Beli di train : {y_train.mean()*100:.1f}%")
print(f"Proporsi kelas Beli di test  : {y_test.mean()*100:.1f}%")


Setelah standardisasi:
y_train — Beli (1): 6,814  |  Tidak Beli (0): 869
y_test  — Beli (1): 1,715   |  Tidak Beli (0): 206

Proporsi kelas Beli di train : 88.7%
Proporsi kelas Beli di test  : 89.3%


## 4. Teori Naïve Bayes

### Formula Bayes
$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

| Simbol | Arti |
|--------|------|
| $P(C\|X)$ | Probabilitas kelas C diberikan fitur X *(posterior)* |
| $P(X\|C)$ | Probabilitas fitur X jika kelasnya C *(likelihood)* |
| $P(C)$ | Probabilitas kelas C *(prior)* |
| $P(X)$ | Probabilitas fitur X *(evidence, konstan)* |

### Contoh Konkret dengan Data Ini
Misalnya ulasan mengandung kata **"bagus"** dan **"cepat"**:

> P(Beli | bagus, cepat) ∝ P(bagus|Beli) × P(cepat|Beli) × P(Beli)

Model membandingkan nilai ini untuk kelas **Beli** vs **Tidak Beli**, lalu pilih yang lebih besar.

### Dua Varian yang Diuji

| Varian | Cara Kerja | Keunggulan |
|--------|-----------|-----------|
| **Multinomial NB** | Belajar dari distribusi kata di kelas target | Standar untuk klasifikasi teks |
| **Complement NB** | Belajar dari distribusi kata di kelas *lain* | Lebih baik untuk dataset tidak seimbang |

### Parameter Alpha (Laplace Smoothing)
Mencegah probabilitas = 0 untuk kata yang belum pernah muncul di training.
- `alpha=0.1` → sedikit smoothing (lebih percaya data)
- `alpha=1.0` → smoothing penuh (lebih konservatif)


## 5. Eksperimen 6 Varian Naïve Bayes

Menguji: 2 varian × 3 nilai alpha = **6 model**


In [ ]:
models = {
    "Multinomial NB (alpha=0.1)": MultinomialNB(alpha=0.1),
    "Multinomial NB (alpha=0.5)": MultinomialNB(alpha=0.5),
    "Multinomial NB (alpha=1.0)": MultinomialNB(alpha=1.0),
    "Complement NB  (alpha=0.1)": ComplementNB(alpha=0.1),
    "Complement NB  (alpha=0.5)": ComplementNB(alpha=0.5),
    "Complement NB  (alpha=1.0)": ComplementNB(alpha=1.0),
}

hasil = []

print(f"{'Model':<35} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 85)

for nama, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_proba)

    hasil.append({"Model": nama.strip(), "Accuracy": acc,
                  "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc})
    print(f"{nama:<35} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc:>10.4f}")

hasil_df = pd.DataFrame(hasil).sort_values("F1", ascending=False).reset_index(drop=True)
hasil_df.index += 1


Model                                 Accuracy  Precision     Recall         F1        AUC
-------------------------------------------------------------------------------------
Multinomial NB (alpha=0.1)              0.9334     0.9542     0.9720     0.9630     0.9433
Multinomial NB (alpha=0.5)              0.9292     0.9384     0.9854     0.9613     0.9349
Multinomial NB (alpha=1.0)              0.9146     0.9172     0.9942     0.9541     0.9278
Complement NB  (alpha=0.1)              0.8527     0.9864     0.8466     0.9112     0.9433
Complement NB  (alpha=0.5)              0.8391     0.9875     0.8303     0.9021     0.9349
Complement NB  (alpha=1.0)              0.8584     0.9769     0.8618     0.9157     0.9278


## 6. Tabel Perbandingan Semua Model

In [ ]:
display_df = hasil_df.copy()
for col in ['Accuracy','Precision','Recall','F1','AUC']:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")

print("PERINGKAT MODEL (urut F1-Score tertinggi)")
display_df


PERINGKAT MODEL (urut F1-Score tertinggi)


,Model,Accuracy,Precision,Recall,F1,AUC
1,Multinomial NB (alpha=0.1),0.9334,0.9542,0.9720,0.9630,0.9433
2,Multinomial NB (alpha=0.5),0.9292,0.9384,0.9854,0.9613,0.9349
3,Multinomial NB (alpha=1.0),0.9146,0.9172,0.9942,0.9541,0.9278
4,Complement NB (alpha=1.0),0.8584,0.9769,0.8618,0.9157,0.9278
5,Complement NB (alpha=0.1),0.8527,0.9864,0.8466,0.9112,0.9433
6,Complement NB (alpha=0.5),0.8391,0.9875,0.8303,0.9021,0.9349


## 7. Pilih Model Terbaik

In [ ]:
import re as _re

best_row  = hasil_df.iloc[0]
best_name = best_row["Model"]

print("=" * 55)
print("MODEL TERPILIH")
print("=" * 55)
print(f"  Nama       : {best_name}")
print(f"  F1-Score   : {best_row['F1']:.4f}")
print(f"  AUC-ROC    : {best_row['AUC']:.4f}")
print(f"  Accuracy   : {best_row['Accuracy']:.4f}")
print(f"  Precision  : {best_row['Precision']:.4f}")
print(f"  Recall     : {best_row['Recall']:.4f}")

# Instansiasi ulang model terbaik
alpha_val = float(_re.search(r'alpha=(\d+\.\d+)', best_name).group(1))
best_model = ComplementNB(alpha=alpha_val) if "Complement" in best_name else MultinomialNB(alpha=alpha_val)
best_model.fit(X_train, y_train)
print()
print(f"Model dilatih pada {X_train.shape[0]:,} data training dari Najmi.")


MODEL TERPILIH
  Nama       : Multinomial NB (alpha=0.1)
  F1-Score   : 0.9630
  AUC-ROC    : 0.9433
  Accuracy   : 0.9334
  Precision  : 0.9542
  Recall     : 0.9720

Model dilatih pada 7,683 data training dari Najmi.


## 8. Classification Report Lengkap

In [ ]:
y_pred_best = best_model.predict(X_test)

print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred_best,
                             target_names=["Tidak Beli (N)", "Beli (Y)"]))

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(f"                    Pred: Tidak Beli   Pred: Beli")
print(f"  Actual: Tidak Beli       {tn:>5}          {fp:>5}")
print(f"  Actual: Beli             {fn:>5}          {tp:>5}")
print()
print(f"  True Negative  (TN) = {tn:,}")
print(f"  False Positive (FP) = {fp:,}")
print(f"  False Negative (FN) = {fn:,}")
print(f"  True Positive  (TP) = {tp:,}")


CLASSIFICATION REPORT
                precision    recall  f1-score   support

Tidak Beli (N)       0.72      0.61      0.66       206
      Beli (Y)       0.95      0.97      0.96      1715

      accuracy                           0.93      1921
     macro avg       0.84      0.79      0.81      1921
  weighted avg       0.93      0.93      0.93      1921

Confusion Matrix:
                    Pred: Tidak Beli   Pred: Beli
  Actual: Tidak Beli         126             80
  Actual: Beli                48           1667

  True Negative  (TN) = 126
  False Positive (FP) = 80
  False Negative (FN) = 48
  True Positive  (TP) = 1,667


## 9. Cross-Validation 5-Fold

In [ ]:
cv_scores = cross_val_score(best_model, X_train, y_train,
                             cv=5, scoring='f1', n_jobs=-1)

print("CROSS-VALIDATION (5-Fold) — F1-Score")
print("=" * 45)
for i, score in enumerate(cv_scores, 1):
    bar = '█' * int(score * 30)
    print(f"  Fold {i}: {score:.4f}  {bar}")
print()
print(f"  Rata-rata : {cv_scores.mean():.4f}")
print(f"  Std Dev   : {cv_scores.std():.4f}")
if cv_scores.std() < 0.02:
    print("  ✅ Model STABIL — std dev rendah, tidak overfitting")
else:
    print("  ⚠️  Pertimbangkan tuning lebih lanjut")


CROSS-VALIDATION (5-Fold) — F1-Score
  Fold 1: 0.9636  ████████████████████████████
  Fold 2: 0.9586  ████████████████████████████
  Fold 3: 0.9532  ████████████████████████████
  Fold 4: 0.9534  ████████████████████████████
  Fold 5: 0.9624  ████████████████████████████

  Rata-rata : 0.9582
  Std Dev   : 0.0044
  ✅ Model STABIL — std dev rendah, tidak overfitting


## 10. Simpan Model ke File `.pkl`

Model disimpan agar bisa dipakai ulang oleh **Syaila** (evaluasi) dan **Dwiki** (dashboard) tanpa perlu training ulang.


In [ ]:
with open("model_naive_bayes.pkl", "wb") as f:
    pickle.dump(best_model, f)
print("✅ model_naive_bayes.pkl  — tersimpan")

hasil_df.to_excel("hasil_eksperimen_naive_bayes.xlsx", index=False)
print("✅ hasil_eksperimen_naive_bayes.xlsx — tersimpan")

print()
print("Cara load model nanti:")
print("  import pickle")
print("  model = pickle.load(open('model_naive_bayes.pkl', 'rb'))")
print("  y_pred = model.predict(X_test_baru)")


✅ model_naive_bayes.pkl  — tersimpan
✅ hasil_eksperimen_naive_bayes.xlsx — tersimpan

Cara load model nanti:
  import pickle
  model = pickle.load(open('model_naive_bayes.pkl', 'rb'))
  y_pred = model.predict(X_test_baru)
